# NanoLLM v2 - Training Notebook
Train a ~50M parameter language model from scratch.

**Steps:**
1. Clone repository
2. Install dependencies
3. Verify GPU
4. Preprocess dataset
5. Train tokenizer
6. Run pre-flight checks
7. Start training
8. Evaluate
9. Run inference

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/CodewithMubasher/NanoLLM-v2.git
%cd NanoLLM-v2

## Step 2: Install Dependencies

In [ ]:
!pip install -r requirements.txt -q

## Step 3: Verify GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 4: Train Tokenizer

In [ ]:
from tokenizer import train_tokenizer

# Create a small corpus for tokenizer training
import json
corpus_lines = []

# Add some general English text
sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Python is a programming language that lets you work quickly.",
    "Machine learning is a subset of artificial intelligence.",
    "The capital of France is Paris.",
    "Water is composed of hydrogen and oxygen.",
] * 1000

with open("data/tokenizer/corpus.txt", "w") as f:
    for text in sample_texts:
        f.write(text + "\n")

train_tokenizer(
    ["data/tokenizer/corpus.txt"],
    "data/tokenizer/tokenizer.json",
    vocab_size=16384
)

## Step 5: Download and Preprocess Dataset

In [ ]:
from preprocess import preprocess_all

preprocess_all()

## Step 6: Run Pre-flight Checks

In [ ]:
from config import Config
from train import run_preflight

config = Config()
run_preflight(config)

## Step 7: Start Training
**Run this cell to start training (~2 hours on T4)**

In [ ]:
from config import Config
from train import train

config = Config()
config.train.max_steps = 60000
config.train.batch_size = 12
config.train.gradient_accumulation_steps = 5

model = train(config)

## Step 8: Evaluate

In [ ]:
from evaluate import evaluate

evaluate("checkpoints/best.pt")

## Step 9: Interactive Inference

In [ ]:
from inference import load_model, generate
from tokenizer import load_tokenizer

model, config, step, best_loss = load_model("checkpoints/best.pt")
tokenizer = load_tokenizer("data/tokenizer/tokenizer.json")

def ask(question):
    response = generate(model, tokenizer, question, "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Q: {question}")
    print(f"A: {response}\n")

ask("What is a computer?")
ask("Why is the sky blue?")
ask("Explain gravity simply.")